In [ ]:
pip install opencv-python

In [ ]:
pip install inference-sdk

In [ ]:
import cv2
import os
from inference_sdk import InferenceHTTPClient
from dotenv import load_dotenv


load_dotenv()

INPUT_DIR = "/input_images"
OUTPUT_DIR = "/output_images"


API_KEY = os.getenv("ROBOFLOW_API_KEY")


if not API_KEY:
    raise ValueError("Błąd: Nie znaleziono klucza API! Upewnij się, że masz plik .env lub ustawione GitHub Secrets.")

MODEL_ID = "roi_detection-82ro3/3"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

CLIENT = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key=API_KEY
)

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png')


print(f"Scanning folder: {INPUT_DIR}")

for filename in os.listdir(INPUT_DIR):
    if not filename.lower().endswith(VALID_EXTENSIONS):
        continue

    input_path = os.path.join(INPUT_DIR, filename)
    print(f"\nProcessing: {filename}...")


    try:
        api_result = CLIENT.infer(input_path, model_id=MODEL_ID)
    except Exception as e:
        print(f"API error for file {filename}: {e}")
        continue


    image = cv2.imread(input_path)

    if image is None:
        print(f"Error: Failed to load image from path: {input_path}")
        continue


    img_height, img_width = image.shape[:2]

    for i, prediction in enumerate(api_result['predictions']):
        x_center = prediction['x']
        y_center = prediction['y']
        width = prediction['width']
        height = prediction['height']


        x1 = int(x_center - (width / 2))
        y1 = int(y_center - (height / 2))
        x2 = int(x_center + (width / 2))
        y2 = int(y_center + (height / 2))

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img_width, x2)
        y2 = min(img_height, y2)


        if x2 <= x1 or y2 <= y1:
            continue


        cropped_image = image[y1:y2, x1:x2]

        name, ext = os.path.splitext(filename)

        crop_filename = f"{name}_crop_{i}{ext}"
        output_path = os.path.join(OUTPUT_DIR, crop_filename)

        cv2.imwrite(output_path, cropped_image)
        print(f" Saved crop: {output_path}")

print("\nFinished processing and cropping all images!")